# Data Types (dtype) and Type Casting

## Definition
Every NumPy array has a **dtype** (data type) that defines how each element is stored in memory.
Choosing the right dtype is critical in AI/ML:
- `float32` is the standard for neural network weights (balances precision + speed + memory)
- `int8` / `float16` used in model quantization for inference speedup
- `bool` for masks in attention mechanisms

## Common dtypes

| dtype | Bits | Range / Precision |
|-------|------|-------------------|
| `int8` | 8 | -128 to 127 |
| `int16` | 16 | -32768 to 32767 |
| `int32` | 32 | ~-2.1B to 2.1B |
| `int64` | 64 | large integers |
| `float16` | 16 | ~±65504, low precision |
| `float32` | 32 | ~7 decimal digits |
| `float64` | 64 | ~15 decimal digits |
| `bool` | 8 | True / False |
| `complex64` | 64 | 2×float32 |
| `complex128` | 128 | 2×float64 |

In [ ]:
import numpy as np

# Creating arrays with explicit dtypes
i32  = np.array([1, 2, 3], dtype=np.int32)
f32  = np.array([1.0, 2.0], dtype=np.float32)
f64  = np.array([1.0, 2.0], dtype=np.float64)
b    = np.array([True, False, True], dtype=np.bool_)
c64  = np.array([1+2j, 3+4j], dtype=np.complex64)

for arr in [i32, f32, f64, b, c64]:
    print(f'dtype={arr.dtype:12}  itemsize={arr.itemsize}  values={arr}')

In [ ]:
# dtype info: range, precision
for dt in [np.int8, np.int16, np.int32, np.int64]:
    info = np.iinfo(dt)
    print(f'{str(dt):12}  min={info.min:20}  max={info.max}')

print()
for dt in [np.float16, np.float32, np.float64]:
    info = np.finfo(dt)
    print(f'{str(dt):12}  min={info.min:.3e}  max={info.max:.3e}  eps={info.eps:.3e}')

In [ ]:
# Type Casting: astype()
arr = np.array([1.7, 2.9, 3.1])
print('Original float64:', arr)

as_int   = arr.astype(np.int32)    # truncates (not rounds!)
as_float = as_int.astype(np.float32)
as_str   = arr.astype(str)

print('Cast to int32 (truncated):', as_int)
print('Cast back to float32:', as_float)
print('Cast to str:', as_str)

In [ ]:
# Automatic type promotion when mixing dtypes
a = np.array([1, 2, 3], dtype=np.int32)
b = np.array([1.5, 2.5, 3.5], dtype=np.float64)
c = a + b  # int32 + float64 -> float64
print('int32 + float64 =', c.dtype, c)

d = np.array([1], dtype=np.float16)
e = np.array([1], dtype=np.float32)
print('float16 + float32 =', (d + e).dtype)

In [ ]:
# Memory savings with lower precision
size = (1000, 1000)
f64 = np.ones(size, dtype=np.float64)
f32 = np.ones(size, dtype=np.float32)
f16 = np.ones(size, dtype=np.float16)

print(f'float64: {f64.nbytes/1024:.0f} KB')
print(f'float32: {f32.nbytes/1024:.0f} KB')
print(f'float16: {f16.nbytes/1024:.0f} KB')

In [ ]:
# EXCEPTION 1: Casting string that can't convert
try:
    bad = np.array(['hello', 'world']).astype(np.float32)
except ValueError as e:
    print('ValueError:', e)

# EXCEPTION 2: Overflow when casting — silent!
big = np.array([1000], dtype=np.int32)
small = big.astype(np.int8)   # 1000 overflows int8 range [-128, 127]
print('int32 1000 cast to int8:', small)  # wraps!

# EXCEPTION 3: Complex to real (imaginary part discarded with warning)
import warnings
c = np.array([1+2j], dtype=np.complex64)
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter('always')
    real = c.astype(np.float32)
    if w:
        print('Warning:', w[0].message)
print('Complex to real:', real)